# Libs

In [ ]:
import sys
sys.path.append("../libs/")
sys.path.append("../")

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset, random_split

from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

import futurai_ppd as ppd
import futurai_utils as utils
from futurai_ml_dev import FuturaiML

DIR_DATA = os.getcwd() + "/data/"

# Load Data

In [ ]:
process_name = "Refinador - Rosca Extração"
timestamp="TIMESTAMP"

list_colomuns_drop = ["espessura_receita_prensa"]

df_dataset = ppd.load_dataset_principal(DIR_DATA+process_name+".csv", list_colomuns_drop, timestamp, dropna=True, use_chunks=True, chunksize=10000)
df_dataset

# Set ON/OFF var

In [ ]:
pre_process = []
pp_var_ref_desligado = "R_2314S_Motor"
pp_valor_ref_desligado = 400
pp_tempo_ref_desligado = 0
pp_pre_corte_transitorio = 60
pp_pos_corte_transitorio = 210
pre_process.append(  
{
   "after_cut": pp_pos_corte_transitorio,
   "interval_off": pp_tempo_ref_desligado,
   "limit_off": pp_valor_ref_desligado,
   "pre_cut": pp_pre_corte_transitorio,
   "variable_off": pp_var_ref_desligado
  })

# Preprocessing Data

In [ ]:
print(f"Dataset shape before preprocessing: {df_dataset.shape}")
for pro in pre_process:
    df_dataset,_,_ = ppd.drop_transitorio_desligado(df_dataset,pro["variable_off"],pro["limit_off"],pro["interval_off"],timestamp,pre_corte=pro["pre_cut"],pos_corte=pro["after_cut"])
df_dataset.reset_index(inplace=True, drop=True)
print(f"Dataset shape after preprocessing: {df_dataset.shape}")

# TAGs and descriptions

In [ ]:
df_sistema, df_sistema_drop =  ppd.set_tags_config(df_dataset,DIR_DATA+process_name+"_subsistema.csv")
df_sistema

# Select training periods (Atual)

In [ ]:
fig = utils.select_training_period(df_dataset, timestamp)
fig.show()

# Data Train

## Period 1

In [ ]:
start_date_train1 = pd.to_datetime("2025-07-10 00:00:00")
end_date_train1 = pd.to_datetime("2025-07-15 00:00:00")

mask = (df_dataset[timestamp] >= start_date_train1) & (df_dataset[timestamp] <= end_date_train1)
df_train1 = df_dataset.loc[mask]

eixoX_train1 = df_train1[timestamp]
df_train1 = df_train1.drop(timestamp,axis=1)

## Period 2

In [ ]:
start_date_train2 = pd.to_datetime("2025-10-21 20:00:00")
end_date_train2 = pd.to_datetime("2025-10-23 09:00:00")

mask = (df_dataset[timestamp] >= start_date_train2) & (df_dataset[timestamp] <= end_date_train2)
df_train2 = df_dataset.loc[mask]

eixoX_train2 = df_train2[timestamp]
df_train2 = df_train2.drop(timestamp,axis=1)

df_train = pd.concat([df_train1, df_train2], ignore_index=True)
eixoX_train = pd.concat([eixoX_train1, eixoX_train2], ignore_index=True)

# Data Test

In [ ]:
start_date = pd.to_datetime("2025-07-01 00:00:00")
end_date = pd.to_datetime("2025-11-30 19:21:00")

mask = (df_dataset[timestamp] > start_date) & (df_dataset[timestamp] <= end_date)
df_test = df_dataset.loc[mask]

eixoX_test = df_test[timestamp]
df_test_aux = df_test.copy()
df_test_aux.set_index(timestamp, inplace=True)
df_test = df_test.drop(timestamp,axis=1)
df_test.reset_index(inplace=True, drop=True)
print(f"Data Test Shape: {df_test.shape}")

# Anomaly Detection

## PCA $\phi$(T$^2$ + Q)

In [ ]:
# Instacinamento da Classe
gain = 2.5
nc = 0
futurai = FuturaiML(nc,gain)
futurai.fit(df_train)

print("T²: {:.2f}".format(futurai.t2_lim))
print("Q: {:.2f}".format(futurai.q_lim))
print("Phi: {:.2f}".format(futurai.phi_lim))
print("Componentes: {:}".format(futurai.nc))

In [ ]:
result = futurai.predict(df_test,eixoX_test)

fig_all_period, _ = utils.dev_graph_predict(result["phi"], result["timestamp"], futurai.phi_lim, " ", start_date, end_date, list_periods=None, plot_anomalies=False)
fig_all_period.show()

## ANN-Based AutoEncoder

Tziolas, T.; Papageorgiou, K.; Theodosiou, T.; Papageorgiou, E.; Mastos, T.; Papadopoulos, A. Autoencoders for Anomaly Detection in an Industrial Multivariate Time Series Dataset. Eng. Proc. 2022, 18, 23. https://doi.org/10.3390/engproc2022018023

In [ ]:
class ANN_Autoencoder(nn.Module):
    def __init__(self, seq_len=60, n_features=3):
        super(ANN_Autoencoder, self).__init__()
        
        self.seq_len = seq_len
        self.n_features = n_features
        
        # Encoder
        # "Input (128-64-32)-flatten-1024-latent space (128)"
        self.enc_layer1 = nn.Linear(n_features, 128)
        self.enc_layer2 = nn.Linear(128, 64)
        self.enc_layer3 = nn.Linear(64, 32)
        
        # Flatten
        self.flatten_dim = seq_len * 32
        
        self.enc_layer4 = nn.Linear(self.flatten_dim, 1024)
        self.latent_layer = nn.Linear(1024, 128) # Latent Space
        
        # Decoder
        # "(1024-64-32)-reshape (64-128)-output" 
        self.dec_layer1 = nn.Linear(128, 1024)
        self.dec_layer2 = nn.Linear(1024, self.flatten_dim)
        
        # Reshape
        self.dec_layer3 = nn.Linear(32, 64)
        self.dec_layer4 = nn.Linear(64, 128)
        self.output_layer = nn.Linear(128, n_features)
        
        self.relu = nn.ReLU()

    def forward(self, x):
        # x shape: [batch_size, 60, n_features]
        
        # Encoder Pass
        x = self.relu(self.enc_layer1(x))
        x = self.relu(self.enc_layer2(x))
        x = self.relu(self.enc_layer3(x)) # Output: [batch, 60, 32]
        
        # Flatten
        x = x.view(x.size(0), -1)         # Output: [batch, 1920]
        
        x = self.relu(self.enc_layer4(x))
        latent = self.relu(self.latent_layer(x)) # Output: [batch, 128]
        
        # Decoder Pass
        x = self.relu(self.dec_layer1(latent))
        x = self.relu(self.dec_layer2(x)) # Output: [batch, 1920]
        
        # Reshape (unflatten)
        x = x.view(x.size(0), self.seq_len, 32) # Output: [batch, 60, 32]
        
        x = self.relu(self.dec_layer3(x))
        x = self.relu(self.dec_layer4(x))
        
        reconstruction = self.output_layer(x) # Output: [batch, 60, n_features]
        
        return reconstruction

In [ ]:
SEQ_LEN = 1
STRIDE = 1
BATCH_SIZE = 128 # Valor padrão de mercado (32 a 128)
LR = 1e-3
EPOCHS = 200
PATIENCE = 1000    # se não melhorar em 100 épocas para o treino

scaler = MinMaxScaler()
df_train_scaled = scaler.fit_transform(df_train)

def reshape_data(data, seq_len, stride=1):
    """
    (N, features) -> (Num_Janelas, seq_len, features)
    Parâmetros:
      stride (int): passo das janelas 
                    Se stride=1 -> máxima sobreposição
                    Se stride=seq_len -> sem sobreposição
    """
    xs = []
    for i in range(0, len(data) - seq_len + 1, stride):
        x = data[i:(i + seq_len)]
        xs.append(x)
    return np.array(xs)

X_train_full = reshape_data(df_train_scaled, SEQ_LEN, STRIDE)
tensor_data = torch.tensor(X_train_full, dtype=torch.float32)

train_size = int(0.9 * len(tensor_data))

train_tensor = tensor_data[:train_size]
val_tensor = tensor_data[train_size:]

train_dataset = TensorDataset(train_tensor)
val_dataset = TensorDataset(val_tensor)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

model_ann = ANN_Autoencoder(seq_len=SEQ_LEN, n_features=df_train.shape[1])
criterion = nn.MSELoss() ## L1Loss || MSELoss
optimizer = optim.Adam(model_ann.parameters(), lr=LR)

best_val_loss = float('inf')
patience_counter = 0

history_loss = []
val_history_loss = []

for epoch in range(EPOCHS):
    model_ann.train()
    train_loss = 0
    for inputs in train_loader:
        if isinstance(inputs, list): inputs = inputs[0]
        optimizer.zero_grad()
        reconstructions = model_ann(inputs)
        loss = criterion(reconstructions, inputs)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    
    avg_train_loss = train_loss / len(train_loader)
    history_loss.append(avg_train_loss)
    
    model_ann.eval()
    val_loss = 0
    with torch.no_grad():
        for inputs in val_loader:
            if isinstance(inputs, list): inputs = inputs[0]
            reconstructions = model_ann(inputs)
            loss = criterion(reconstructions, inputs)
            val_loss += loss.item()
            
    avg_val_loss = val_loss / len(val_loader)
    val_history_loss.append(avg_val_loss)
    
    # Log
    if epoch % 10 == 0 or epoch == EPOCHS:
        print(f"Epoch [{epoch}/{EPOCHS}] | Train Loss: {avg_train_loss:.6f} | Val Loss: {avg_val_loss:.6f}")
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        # Salvar o melhor modelo
        torch.save(model_ann.state_dict(), 'best_model.pth')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Parada antecipada na época {epoch}. Melhor Val Loss: {best_val_loss:.6f}")
            # Recarregar o melhor modelo
            model_ann.load_state_dict(torch.load('best_model.pth'))
            break

In [ ]:
model_ann.eval()
all_mae_losses = []

with torch.no_grad():
    for batch in train_loader:
        inputs = batch[0]
        
        reconstructions = model_ann(inputs)
        
        # Calcula MAE por amostra -> Result shape: [batch_size]
        batch_loss = torch.mean(torch.abs(reconstructions - inputs), dim=[1, 2])
        
        all_mae_losses.extend(batch_loss.cpu().numpy())
        
train_mae_loss = np.array(all_mae_losses)
THRESHOLD = round(np.mean(train_mae_loss) + (20 * np.std(train_mae_loss)), 2)

print(f"Média do Erro: {np.mean(train_mae_loss):.6f}")
print(f"Desvio Padrão: {np.std(train_mae_loss):.6f}")
print(f"Anomaly Threshold: {THRESHOLD:.2f}")

In [ ]:
def predict(df_test, modelo, scaler, threshold, batch_size=32):
    
    X_test_scaled = scaler.transform(df_test.values)

    seq_len = modelo.seq_len
    sequences = reshape_data(X_test_scaled, SEQ_LEN, STRIDE)
    
    if len(sequences) == 0:
        return np.array([]), np.array([])

    dataset = TensorDataset(torch.tensor(sequences, dtype=torch.float32))
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    all_losses = []
    all_preds = []

    modelo.eval()
    with torch.no_grad():
        for batch in loader:
            inputs = batch[0]
            
            # Forward pass
            reconstruction = modelo(inputs)
            
            # MAE por janela
            batch_loss = torch.mean(torch.abs(reconstruction - inputs), dim=[1, 2])
            all_losses.append(batch_loss.numpy())

    final_losses = np.concatenate(all_losses)
    
    return final_losses

erros = predict(df_test, model_ann, scaler, THRESHOLD, batch_size=BATCH_SIZE)

In [ ]:
def plot_anomalies_plotly(df_test, erros, threshold, seq_len=60, stride=1):
    if stride == 1:
        timestamps = df_test.index[seq_len-1:]
    else:
        timestamps = df_test.index[seq_len-1::stride]

    min_len = min(len(timestamps), len(erros))
    timestamps = timestamps[:min_len]
    erros = erros[:min_len]

    plot_df = pd.DataFrame({'erro': erros}, index=timestamps)
    if not isinstance(plot_df.index, pd.DatetimeIndex):
        plot_df.index = pd.to_datetime(plot_df.index)

    freq = f'{stride}min' 
    try:
        plot_df_resampled = plot_df.asfreq(freq)
        plot_df_resampled.fillna(0, inplace=True)
    except ValueError:
        print(" --- Duplicate Indexes ---")
        plot_df_resampled = plot_df

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=plot_df_resampled.index,
        y=plot_df_resampled['erro'],
        mode='lines',
        name='Erro de Reconstrução',
        fill="tozeroy", 
        line=dict(color="#0F293A"),
        connectgaps=False
    ))
    
    fig.add_hline(
        y=threshold, 
        name='Limiar',
        line_dash="dash", 
        line_color="red", 
        line_width=2
    )
    
    fig.update_layout(
        xaxis_title='Data/Hora',
        yaxis_title='Erro Médio Absoluto (MAE)',
        template='plotly_white',
        hovermode="x unified",
        showlegend=True
    )

    fig.show()
    
plot_anomalies_plotly(df_test_aux, erros, THRESHOLD, seq_len=SEQ_LEN, stride=STRIDE)

## LSTM-Based AutoEncoder

Tziolas, T.; Papageorgiou, K.; Theodosiou, T.; Papageorgiou, E.; Mastos, T.; Papadopoulos, A. Autoencoders for Anomaly Detection in an Industrial Multivariate Time Series Dataset. Eng. Proc. 2022, 18, 23. https://doi.org/10.3390/engproc2022018023

In [ ]:
class LSTM_Autoencoder(nn.Module):
    def __init__(self, seq_len=60, n_features=3, hidden_dim=128):
        super(LSTM_Autoencoder, self).__init__()
        
        self.seq_len = seq_len
        self.n_features = n_features
        self.hidden_dim = hidden_dim
        
        # --- Encoder ---
        # Input: [batch, 60, 3] -> Output: [batch, 60, 128]
        self.encoder_l1 = nn.LSTM(
            input_size=n_features, 
            hidden_size=hidden_dim, 
            batch_first=True
        )
        
        # Input: [batch, 60, 128] -> Output (Last Hidden): [batch, 128]
        self.encoder_l2 = nn.LSTM(
            input_size=hidden_dim, 
            hidden_size=hidden_dim, 
            batch_first=True
        )
        
        # --- Decoder ---
        # Input: [batch, 60, 128] -> Output: [batch, 60, 128]
        self.decoder_l1 = nn.LSTM(
            input_size=hidden_dim, 
            hidden_size=hidden_dim, 
            batch_first=True
        )
        
        # Camada 2 (Saída): Reconstrói as features originais
        # Input: [batch, 60, 128] -> Output: [batch, 60, 3]
        self.decoder_l2 = nn.LSTM(
            input_size=hidden_dim, 
            hidden_size=n_features, 
            batch_first=True
        )

    def forward(self, x):
        # x shape: [batch, 60, 3]
        
        # Encoder Pass
        # save enc_out1 for Skip Connection
        enc_out1, _ = self.encoder_l1(x) # shape: [batch, 60, 128]
        
        # Segunda camada LSTM apenas o último estado oculto
        _, (hidden, _) = self.encoder_l2(enc_out1)
        latent = hidden[-1] # shape: [batch, 128]
        
        # Repeat Vector
        # Repete o vetor latente 60 vezes para corresponder a sequência temporal
        # [batch, 128] -> [batch, 60, 128]
        repeat_vect = latent.unsqueeze(1).repeat(1, self.seq_len, 1)
        
        # Skip Connection
        decoder_input = enc_out1 + repeat_vect 
        
        # Decoder Pass
        dec_out1, _ = self.decoder_l1(decoder_input)
        
        # Camada final de reconstrução
        reconstruction, _ = self.decoder_l2(dec_out1)
        
        return reconstruction

In [ ]:
SEQ_LEN = 120
STRIDE = 1
BATCH_SIZE = 128 # Valor padrão de mercado (32 a 128)
LR = 1e-3
EPOCHS = 50
PATIENCE = 1000    # se não melhorar em 100 épocas para o treino

scaler = MinMaxScaler(feature_range=(-1, 1))
df_train_scaled = scaler.fit_transform(df_train)

def reshape_data(data, seq_len, stride=1):
    """
    (N, features) -> (Num_Janelas, seq_len, features)
    Parâmetros:
      stride (int): passo das janelas 
                    Se stride=1 -> máxima sobreposição
                    Se stride=seq_len -> sem sobreposição
    """
    xs = []
    for i in range(0, len(data) - seq_len + 1, stride):
        x = data[i:(i + seq_len)]
        xs.append(x)
    return np.array(xs)

X_train_full = reshape_data(df_train_scaled, SEQ_LEN, STRIDE)
tensor_data = torch.tensor(X_train_full, dtype=torch.float32)

train_size = int(0.9 * len(tensor_data))

train_tensor = tensor_data[:train_size]
val_tensor = tensor_data[train_size:]

train_dataset = TensorDataset(train_tensor)
val_dataset = TensorDataset(val_tensor)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

model_lstm = LSTM_Autoencoder(seq_len=SEQ_LEN, n_features=df_train.shape[1], hidden_dim=128)
criterion = nn.MSELoss() ## L1Loss || MSELoss
optimizer = optim.Adam(model_lstm.parameters(), lr=LR)

best_val_loss = float('inf')
patience_counter = 0

history_loss = []
val_history_loss = []

for epoch in range(EPOCHS):
    model_lstm.train()
    train_loss = 0
    for inputs in train_loader:
        if isinstance(inputs, list): inputs = inputs[0]
        optimizer.zero_grad()
        reconstructions = model_lstm(inputs)
        loss = criterion(reconstructions, inputs)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    
    avg_train_loss = train_loss / len(train_loader)
    history_loss.append(avg_train_loss)
    
    model_lstm.eval()
    val_loss = 0
    with torch.no_grad():
        for inputs in val_loader:
            if isinstance(inputs, list): inputs = inputs[0]
            reconstructions = model_lstm(inputs)
            loss = criterion(reconstructions, inputs)
            val_loss += loss.item()
            
    avg_val_loss = val_loss / len(val_loader)
    val_history_loss.append(avg_val_loss)
    
    # Log
    if epoch % 10 == 0 or epoch == EPOCHS:
        print(f"Epoch [{epoch}/{EPOCHS}] | Train Loss: {avg_train_loss:.6f} | Val Loss: {avg_val_loss:.6f}")
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        # Salvar o melhor modelo
        torch.save(model_lstm.state_dict(), 'best_model.pth')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Parada antecipada na época {epoch}. Melhor Val Loss: {best_val_loss:.6f}")
            # Recarregar o melhor modelo
            model_lstm.load_state_dict(torch.load('best_model.pth'))
            break

In [ ]:
model_lstm.eval()
all_mae_losses = []

with torch.no_grad():
    for batch in train_loader:
        inputs = batch[0]
        
        reconstructions = model_lstm(inputs)
        
        # Calcula MAE por amostra -> Result shape: [batch_size]
        batch_loss = torch.mean(torch.abs(reconstructions - inputs), dim=[1, 2])
        
        all_mae_losses.extend(batch_loss.cpu().numpy())
        
train_mae_loss = np.array(all_mae_losses)
THRESHOLD = round(np.mean(train_mae_loss) + (20 * np.std(train_mae_loss)), 2)

print(f"Média do Erro: {np.mean(train_mae_loss):.6f}")
print(f"Desvio Padrão: {np.std(train_mae_loss):.6f}")
print(f"Anomaly Threshold: {THRESHOLD:.2f}")

In [ ]:
def predict(df_test, modelo, scaler, threshold, batch_size=32):
    
    X_test_scaled = scaler.transform(df_test.values)

    seq_len = modelo.seq_len
    sequences = reshape_data(X_test_scaled, SEQ_LEN, STRIDE)
    
    if len(sequences) == 0:
        return np.array([]), np.array([])

    dataset = TensorDataset(torch.tensor(sequences, dtype=torch.float32))
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    all_losses = []
    all_preds = []

    modelo.eval()
    with torch.no_grad():
        for batch in loader:
            inputs = batch[0]
            
            # Forward pass
            reconstruction = modelo(inputs)
            
            # MAE por janela
            batch_loss = torch.mean(torch.abs(reconstruction - inputs), dim=[1, 2])
            all_losses.append(batch_loss.numpy())

    final_losses = np.concatenate(all_losses)
    
    return final_losses

erros = predict(df_test, model_lstm, scaler, THRESHOLD, batch_size=BATCH_SIZE)

In [ ]:
def plot_anomalies_plotly(df_test, erros, threshold, seq_len=60, stride=1):
    if stride == 1:
        timestamps = df_test.index[seq_len-1:]
    else:
        timestamps = df_test.index[seq_len-1::stride]

    min_len = min(len(timestamps), len(erros))
    timestamps = timestamps[:min_len]
    erros = erros[:min_len]

    plot_df = pd.DataFrame({'erro': erros}, index=timestamps)
    if not isinstance(plot_df.index, pd.DatetimeIndex):
        plot_df.index = pd.to_datetime(plot_df.index)

    freq = f'{stride}min' 
    try:
        plot_df_resampled = plot_df.asfreq(freq)
        plot_df_resampled.fillna(0, inplace=True)
    except ValueError:
        print(" --- Duplicate Indexes ---")
        plot_df_resampled = plot_df

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=plot_df_resampled.index,
        y=plot_df_resampled['erro'],
        mode='lines',
        name='Erro de Reconstrução',
        fill="tozeroy", 
        line=dict(color="#0F293A"),
        connectgaps=False
    ))
    
    fig.add_hline(
        y=threshold, 
        name='Limiar',
        line_dash="dash", 
        line_color="red", 
        line_width=2
    )
    
    fig.update_layout(
        xaxis_title='Data/Hora',
        yaxis_title='Erro Médio Absoluto (MAE)',
        template='plotly_white',
        hovermode="x unified",
        showlegend=True
    )

    fig.show()
    
plot_anomalies_plotly(df_test_aux, erros, THRESHOLD, seq_len=SEQ_LEN, stride=STRIDE)

## CNN-Based AutoEncoder

Tziolas, T.; Papageorgiou, K.; Theodosiou, T.; Papageorgiou, E.; Mastos, T.; Papadopoulos, A. Autoencoders for Anomaly Detection in an Industrial Multivariate Time Series Dataset. Eng. Proc. 2022, 18, 23. https://doi.org/10.3390/engproc2022018023

In [ ]:
class CNN_Autoencoder(nn.Module):
    def __init__(self, seq_len=60, n_features=3):
        super(CNN_Autoencoder, self).__init__()
        self.seq_len = seq_len
        self.n_features = n_features

        # --- ENCODER ---
        # Layer 1: Conv1D 64, kernel 15, stride 3
        self.enc_conv1 = nn.Conv1d(in_channels=n_features, out_channels=64, kernel_size=15, stride=3, padding=7)
        
        # Layer 2: Conv1D 32, kernel 9, stride 2
        self.enc_conv2 = nn.Conv1d(64, 32, kernel_size=9, stride=2, padding=4)
        
        # Layer 3: Conv1D 16, kernel 7, stride 2
        self.enc_conv3 = nn.Conv1d(32, 16, kernel_size=7, stride=2, padding=3)
        
        # Layer 4: Conv1D 8, kernel 3, stride 1
        self.enc_conv4 = nn.Conv1d(16, 8, kernel_size=3, stride=1, padding=1)
        
        # Max Pooling: Size 2, Stride 2
        self.max_pool = nn.MaxPool1d(kernel_size=2, stride=2)

        # --- DECODER ---
        # Utiliza Transpose Convolutions para Up-sampling 
        
        # Layer 1 Decoder: TranspConv 16, kernel 3, stride 2
        self.dec_trans1 = nn.ConvTranspose1d(8, 16, kernel_size=3, stride=2, padding=1, output_padding=1)
        
        # Layer 2 Decoder: TranspConv 32, kernel 7, stride 2
        self.dec_trans2 = nn.ConvTranspose1d(16, 32, kernel_size=7, stride=2, padding=3, output_padding=1)
        
        # Layer 3 Decoder: TranspConv 64, kernel 9, stride 2
        self.dec_trans3 = nn.ConvTranspose1d(32, 64, kernel_size=9, stride=2, padding=4, output_padding=1)
        
        # Layer 4 Decoder: TranspConv 64, kernel 15, stride 3
        self.dec_trans4 = nn.ConvTranspose1d(64, 64, kernel_size=15, stride=3, padding=7, output_padding=1)
        
        # Camada Final: Linear/Conv1d com kernel 1 para voltar as features originais
        self.dec_final = nn.Conv1d(64, n_features, kernel_size=1, stride=1)
        
        self.relu = nn.ReLU()

    def forward(self, x):
        # x input shape: [batch, seq_len=60, features=3]
        
        # 1. Transpor para formato CNN: [batch, 3, 60]
        x = x.permute(0, 2, 1)
        
        # --- Encoder Pass ---
        x = self.relu(self.enc_conv1(x))
        x = self.relu(self.enc_conv2(x))
        x = self.relu(self.enc_conv3(x))
        x = self.relu(self.enc_conv4(x))
        x = self.max_pool(x)
        
        # --- Decoder Pass ---
        x = self.relu(self.dec_trans1(x))
        x = self.relu(self.dec_trans2(x))
        x = self.relu(self.dec_trans3(x))
        x = self.relu(self.dec_trans4(x))
        
        # Output layer (sem ativação se os dados forem normalizados, ou Sigmoid se for 0-1)
        x = self.dec_final(x)
        
        # Ajuste Fino de Tamanho: 
        # Como convoluções com stride em sequências pequenas (60) podem perder pixels exatos,
        # forçamos a interpolação para garantir que a saída seja exatamente 60.
        if x.shape[2] != self.seq_len:
            x = F.interpolate(x, size=self.seq_len, mode='linear', align_corners=False)
        
        # 2. Transpor de volta para formato original: [batch, 60, 3]
        x = x.permute(0, 2, 1)
        
        return x

In [ ]:
SEQ_LEN = 120
STRIDE = 1
BATCH_SIZE = 128 # Valor padrão de mercado (32 a 128)
LR = 1e-3
EPOCHS = 100
PATIENCE = 1000    # se não melhorar em 100 épocas para o treino

scaler = MinMaxScaler()
df_train_scaled = scaler.fit_transform(df_train)

def reshape_data(data, seq_len, stride=1):
    """
    (N, features) -> (Num_Janelas, seq_len, features)
    Parâmetros:
      stride (int): passo das janelas 
                    Se stride=1 -> máxima sobreposição
                    Se stride=seq_len -> sem sobreposição
    """
    xs = []
    for i in range(0, len(data) - seq_len + 1, stride):
        x = data[i:(i + seq_len)]
        xs.append(x)
    return np.array(xs)

X_train_full = reshape_data(df_train_scaled, SEQ_LEN, STRIDE)
tensor_data = torch.tensor(X_train_full, dtype=torch.float32)

train_size = int(0.9 * len(tensor_data))

train_tensor = tensor_data[:train_size]
val_tensor = tensor_data[train_size:]

train_dataset = TensorDataset(train_tensor)
val_dataset = TensorDataset(val_tensor)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

model_cnn = CNN_Autoencoder(seq_len=SEQ_LEN, n_features=df_train.shape[1])
criterion = nn.MSELoss() ## L1Loss || MSELoss
optimizer = optim.Adam(model_cnn.parameters(), lr=LR)

best_val_loss = float('inf')
patience_counter = 0

history_loss = []
val_history_loss = []

for epoch in range(EPOCHS):
    model_cnn.train()
    train_loss = 0
    for inputs in train_loader:
        if isinstance(inputs, list): inputs = inputs[0]
        optimizer.zero_grad()
        reconstructions = model_cnn(inputs)
        loss = criterion(reconstructions, inputs)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    
    avg_train_loss = train_loss / len(train_loader)
    history_loss.append(avg_train_loss)
    
    model_cnn.eval()
    val_loss = 0
    with torch.no_grad():
        for inputs in val_loader:
            if isinstance(inputs, list): inputs = inputs[0]
            reconstructions = model_cnn(inputs)
            loss = criterion(reconstructions, inputs)
            val_loss += loss.item()
            
    avg_val_loss = val_loss / len(val_loader)
    val_history_loss.append(avg_val_loss)
    
    # Log
    if epoch % 10 == 0 or epoch == EPOCHS:
        print(f"Epoch [{epoch}/{EPOCHS}] | Train Loss: {avg_train_loss:.6f} | Val Loss: {avg_val_loss:.6f}")
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        # Salvar o melhor modelo
        torch.save(model_cnn.state_dict(), 'best_model.pth')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Parada antecipada na época {epoch}. Melhor Val Loss: {best_val_loss:.6f}")
            # Recarregar o melhor modelo
            model_cnn.load_state_dict(torch.load('best_model.pth'))
            break

In [ ]:
model_cnn.eval()
all_mae_losses = []

with torch.no_grad():
    for batch in train_loader:
        inputs = batch[0]
        
        reconstructions = model_cnn(inputs)
        
        # Calcula MAE por amostra -> Result shape: [batch_size]
        batch_loss = torch.mean(torch.abs(reconstructions - inputs), dim=[1, 2])
        
        all_mae_losses.extend(batch_loss.cpu().numpy())
        
train_mae_loss = np.array(all_mae_losses)
THRESHOLD = round(np.mean(train_mae_loss) + (10 * np.std(train_mae_loss)), 2)

print(f"Média do Erro: {np.mean(train_mae_loss):.6f}")
print(f"Desvio Padrão: {np.std(train_mae_loss):.6f}")
print(f"Anomaly Threshold: {THRESHOLD:.2f}")

In [ ]:
def predict(df_test, modelo, scaler, threshold, batch_size=32):
    
    X_test_scaled = scaler.transform(df_test.values)

    seq_len = modelo.seq_len
    sequences = reshape_data(X_test_scaled, SEQ_LEN, STRIDE)
    
    if len(sequences) == 0:
        return np.array([]), np.array([])

    dataset = TensorDataset(torch.tensor(sequences, dtype=torch.float32))
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    all_losses = []
    all_preds = []

    modelo.eval()
    with torch.no_grad():
        for batch in loader:
            inputs = batch[0]
            
            # Forward pass
            reconstruction = modelo(inputs)
            
            # MAE por janela
            batch_loss = torch.mean(torch.abs(reconstruction - inputs), dim=[1, 2])
            all_losses.append(batch_loss.numpy())

    final_losses = np.concatenate(all_losses)
    
    return final_losses

erros = predict(df_test, model_cnn, scaler, THRESHOLD, batch_size=BATCH_SIZE)

In [ ]:
def plot_anomalies_plotly(df_test, erros, threshold, seq_len=60, stride=1):
    if stride == 1:
        timestamps = df_test.index[seq_len-1:]
    else:
        timestamps = df_test.index[seq_len-1::stride]

    min_len = min(len(timestamps), len(erros))
    timestamps = timestamps[:min_len]
    erros = erros[:min_len]

    plot_df = pd.DataFrame({'erro': erros}, index=timestamps)
    if not isinstance(plot_df.index, pd.DatetimeIndex):
        plot_df.index = pd.to_datetime(plot_df.index)

    freq = f'{stride}min' 
    try:
        plot_df_resampled = plot_df.asfreq(freq)
        plot_df_resampled.fillna(0, inplace=True)
    except ValueError:
        print(" --- Duplicate Indexes ---")
        plot_df_resampled = plot_df

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=plot_df_resampled.index,
        y=plot_df_resampled['erro'],
        mode='lines',
        name='Erro de Reconstrução',
        fill="tozeroy", 
        line=dict(color="#0F293A"),
        connectgaps=False
    ))
    
    fig.add_hline(
        y=threshold, 
        name='Limiar',
        line_dash="dash", 
        line_color="red", 
        line_width=2
    )
    
    fig.update_layout(
        xaxis_title='Data/Hora',
        yaxis_title='Erro Médio Absoluto (MAE)',
        template='plotly_white',
        hovermode="x unified",
        showlegend=True
    )

    fig.show()
    
plot_anomalies_plotly(df_test_aux, erros, THRESHOLD, seq_len=SEQ_LEN, stride=STRIDE)

## USAD

Audibert, J., Michiardi, P., Guyard, F., Marti, S., Zuluaga, M. A. (2020).
USAD : UnSupervised Anomaly Detection on multivariate time series.
Proceedings of the 26th ACM SIGKDD International Conference on Knowledge Discovery & Data Mining, August 23-27, 2020

In [ ]:
def get_default_device():
    if torch.cuda.is_available():
        return torch.device('cuda')
    else:
        return torch.device('cpu')

device = get_default_device()

class Encoder(nn.Module):
    def __init__(self, in_size, latent_size):
        super().__init__()
        # Arquitetura conforme Apêndice A.4.1
        self.linear1 = nn.Linear(in_size, int(in_size/2))
        self.linear2 = nn.Linear(int(in_size/2), int(in_size/4))
        self.linear3 = nn.Linear(int(in_size/4), latent_size)
        self.relu = nn.ReLU(True)
        
    def forward(self, w):
        out = self.linear1(w)
        out = self.relu(out)
        out = self.linear2(out)
        out = self.relu(out)
        out = self.linear3(out)
        z = self.relu(out)
        return z
    
class Decoder(nn.Module):
    def __init__(self, latent_size, out_size):
        super().__init__()
        # Arquitetura conforme Apêndice A.4.2 [cite: 512-515]
        self.linear1 = nn.Linear(latent_size, int(out_size/4))
        self.linear2 = nn.Linear(int(out_size/4), int(out_size/2))
        self.linear3 = nn.Linear(int(out_size/2), out_size)
        self.relu = nn.ReLU(True)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, z):
        out = self.linear1(z)
        out = self.relu(out)
        out = self.linear2(out)
        out = self.relu(out)
        out = self.linear3(out)
        w = self.sigmoid(out)
        return w
    
class UsadModel(nn.Module):
    def __init__(self, w_size, z_size):
        super().__init__()
        self.encoder = Encoder(w_size, z_size)
        self.decoder1 = Decoder(z_size, w_size)
        self.decoder2 = Decoder(z_size, w_size)
  
    def training_step(self, batch, n):
        # Implementação das Equações (7) e (8)
        z = self.encoder(batch)
        w1 = self.decoder1(z)
        w2 = self.decoder2(z)
        w3 = self.decoder2(self.encoder(w1))
        
        # Loss 1: Treina AE1 para reconstruir e enganar AE2
        loss1 = 1/n * torch.mean((batch - w1)**2) + (1 - 1/n) * torch.mean((batch - w3)**2)
        
        # Loss 2: Treina AE2 para distinguir real de reconstruído
        loss2 = 1/n * torch.mean((batch - w2)**2) - (1 - 1/n) * torch.mean((batch - w3)**2)
        
        return loss1, loss2

    def forward(self, batch):
        # Usado para inferência simples (apenas reconstrução do AE1 e AE2)
        z = self.encoder(batch)
        w1 = self.decoder1(z)
        w2 = self.decoder2(self.encoder(w1))
        return w1, w2

In [ ]:
SEQ_LEN = 120
STRIDE = 1
BATCH_SIZE = 128
LR = 1e-3
EPOCHS = 1 # USAD converge rápido (SWaT - 70 epochs)
Z_SIZE = 64 # Dimensão latente recomendada para SWaT
ALPHA = 0.5
BETA = 0.5

scaler = MinMaxScaler()
df_train_scaled = scaler.fit_transform(df_train)

def reshape_data(data, seq_len, stride=1):
    xs = []
    for i in range(0, len(data) - seq_len + 1, stride):
        x = data[i:(i + seq_len)]
        xs.append(x)
    return np.array(xs)

X_train_full = reshape_data(df_train_scaled, SEQ_LEN, STRIDE)
tensor_data = torch.tensor(X_train_full, dtype=torch.float32)

# Divisão Treino/Validação
train_size = int(0.9 * len(tensor_data))
train_tensor = tensor_data[:train_size]
val_tensor = tensor_data[train_size:]

train_loader = DataLoader(TensorDataset(train_tensor), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(val_tensor), batch_size=BATCH_SIZE, shuffle=False)

# Inicialização do Modelo
W_SIZE = SEQ_LEN * df_train.shape[1] # Tamanho da janela achatada
model_usad = UsadModel(W_SIZE, Z_SIZE)
model_usad = model_usad.to(device)

# Otimizadores (Separados para AE1 e AE2 conforme arquitetura adversarial)
optimizer1 = torch.optim.Adam(list(model_usad.encoder.parameters()) + list(model_usad.decoder1.parameters()), lr=LR)
optimizer2 = torch.optim.Adam(list(model_usad.encoder.parameters()) + list(model_usad.decoder2.parameters()), lr=LR)

# --- Loop de Treinamento ---

history = []
for epoch in range(EPOCHS):
    model_usad.train()
    epoch_loss1 = 0
    epoch_loss2 = 0
    
    for batch in train_loader:
        inputs = batch[0].to(device)
        # Flatten: (Batch, Seq, Feat) -> (Batch, Seq*Feat)
        inputs = inputs.view(inputs.size(0), -1)
        
        # == Treino AE1 ==
        loss1, _ = model_usad.training_step(inputs, epoch + 1)
        optimizer1.zero_grad()
        loss1.backward()
        optimizer1.step()
        
        # == Treino AE2 ==
        _, loss2 = model_usad.training_step(inputs, epoch + 1)
        optimizer2.zero_grad()
        loss2.backward()
        optimizer2.step()
        
        epoch_loss1 += loss1.item()
        epoch_loss2 += loss2.item()
        
    # Validação (Fora do loop de batches para performance)
    model_usad.eval()
    val_loss1 = 0
    val_loss2 = 0
    with torch.no_grad():
        for batch in val_loader:
            inputs = batch[0].to(device)
            inputs = inputs.view(inputs.size(0), -1)
            l1, l2 = model_usad.training_step(inputs, epoch + 1)
            val_loss1 += l1.item()
            val_loss2 += l2.item()
            
    print(f"Epoch [{epoch+1}/{EPOCHS}] | Train Loss1: {epoch_loss1/len(train_loader):.4f} | Val Loss1: {val_loss1/len(val_loader):.4f}")

In [ ]:
def get_usad_scores(model, loader, alpha, beta):
    model.eval()
    scores = []
    with torch.no_grad():
        for batch in loader:
            inputs = batch[0].to(device)
            inputs = inputs.view(inputs.size(0), -1)
            
            w1 = model.decoder1(model.encoder(inputs))
            w2 = model.decoder2(model.encoder(w1))
            
            # Equação 9: Score = Alpha * Diff1 + Beta * Diff2
            diff1 = torch.mean((inputs - w1)**2, axis=1)
            diff2 = torch.mean((inputs - w2)**2, axis=1)
            score = alpha * diff1 + beta * diff2
            
            scores.extend(score.cpu().numpy())
    return np.array(scores)

# Calculando scores no treino para definir o Threshold
train_scores = get_usad_scores(model_usad, train_loader, ALPHA, BETA)
THRESHOLD = np.mean(train_scores) + (3 * np.std(train_scores))

print(f"Média Score Treino: {np.mean(train_scores):.6f}")
print(f"Threshold Definido: {THRESHOLD:.4f}")

In [ ]:
def predict_usad(df_test, model, scaler, seq_len, stride, alpha, beta):
    X_test_scaled = scaler.transform(df_test.values)
    sequences = reshape_data(X_test_scaled, seq_len, stride)
    
    if len(sequences) == 0:
        return np.array([])
        
    dataset = TensorDataset(torch.tensor(sequences, dtype=torch.float32))
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    return get_usad_scores(model, loader, alpha, beta)

# Predição
erros_usad = predict_usad(df_test, model_usad, scaler, SEQ_LEN, STRIDE, ALPHA, BETA)

In [ ]:
def plot_anomalies_plotly(df_test, erros, threshold, seq_len=60, stride=1):
    if stride == 1:
        timestamps = df_test.index[seq_len-1:]
    else:
        timestamps = df_test.index[seq_len-1::stride]

    min_len = min(len(timestamps), len(erros))
    timestamps = timestamps[:min_len]
    erros = erros[:min_len]

    plot_df = pd.DataFrame({'erro': erros}, index=timestamps)
    if not isinstance(plot_df.index, pd.DatetimeIndex):
        plot_df.index = pd.to_datetime(plot_df.index)

    freq = f'{stride}min' 
    try:
        plot_df_resampled = plot_df.asfreq(freq)
        plot_df_resampled.fillna(0, inplace=True)
    except ValueError:
        print(" --- Duplicate Indexes ---")
        plot_df_resampled = plot_df

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=plot_df_resampled.index,
        y=plot_df_resampled['erro'],
        mode='lines',
        name='Erro de Reconstrução',
        fill="tozeroy", 
        line=dict(color="#0F293A"),
        connectgaps=False
    ))
    
    fig.add_hline(
        y=threshold, 
        name='Limiar',
        line_dash="dash", 
        line_color="red", 
        line_width=2
    )
    
    fig.update_layout(
        xaxis_title='Data/Hora',
        yaxis_title='Erro Médio Absoluto (MAE)',
        template='plotly_white',
        hovermode="x unified",
        showlegend=True
    )

    fig.show()

plot_anomalies_plotly(df_test_aux, erros_usad, THRESHOLD, seq_len=SEQ_LEN, stride=STRIDE)

## OmniAnomaly

Ya Su, Youjian Zhao, Chenhao Niu, Rong Liu, Wei Sun, and Dan Pei. 2019. Robust Anomaly Detection for Multivariate Time Series through Stochastic Recurrent Neural Network. In Proceedings of the 25th ACM SIGKDD International Conference on Knowledge Discovery & Data Mining (KDD '19). Association for Computing Machinery, New York, NY, USA, 2828–2837. https://doi.org/10.1145/3292500.3330672

In [ ]:
import os
import time
import torch
import numpy as np
import warnings
from pprint import pprint

# Imports dos seus módulos refatorados
from omni_anomaly.utils import get_data_dim, get_data, save_z
from omni_anomaly.model import OmniAnomaly
from omni_anomaly.training import Trainer
from omni_anomaly.prediction import Predictor
from omni_anomaly.eval_methods import pot_eval, bf_search

# Configuração de dispositivo
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class ExpConfig:
    # Hiperparâmetros do Modelo
    x_dim = df_train.shape[1]
    z_dim = 3
    rnn_num_hidden = 500
    rnn_cell = 'GRU'
    window_length = 60
    dense_dim = 500
    posterior_flow_type = 'nf' # 'nf' ou None
    nf_layers = 20
    
    # Treinamento
    max_epoch = 20
    batch_size = 128
    initial_lr = 0.001
    lr_anneal_factor = 0.5
    lr_anneal_epoch_freq = 10
    grad_clip_norm = 10.0
    l2_reg = 0.0001
    
    # Configurações do OmniAnomaly original
    use_connected_z_q = True
    use_connected_z_p = True
    std_epsilon = 1e-4

    # Teste e Avaliação
    test_n_z = 1
    test_batch_size = 128
    valid_step_freq = 100
    
    # Parâmetros de Avaliação (POT)
    level = 0.01
    
    # Parâmetros de busca BF (Best F1)
    bf_search_min = -400.
    bf_search_max = 400.
    bf_search_step_size = 1.
    
    # Outros
    save_z = False
    get_score_on_dim = False # Se True, retorna score por dimensão, senão soma

config = ExpConfig()

In [ ]:
x_train = df_train.values # Converter para numpy
x_test = df_test.values

# 1. Garantir tipo float32 (essencial para PyTorch)
x_train = x_train.astype(np.float32)
x_test = x_test.astype(np.float32)

config.x_dim = x_train.shape[1]

scaler = MinMaxScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

print(f"Shape Treino: {x_train.shape}")
print(f"Shape Teste: {x_test.shape}")
print(f"Features (x_dim): {config.x_dim}")

In [ ]:
# Instanciar o modelo refatorado (PyTorch)
model = OmniAnomaly(config).to(device)

# Instanciar o Trainer
trainer = Trainer(
    model=model,
    max_epoch=config.max_epoch,
    batch_size=config.batch_size,
    valid_batch_size=config.test_batch_size,
    initial_lr=config.initial_lr,
    lr_anneal_epochs=config.lr_anneal_epoch_freq,
    lr_anneal_factor=config.lr_anneal_factor,
    grad_clip_norm=config.grad_clip_norm,
    l2_reg=config.l2_reg,
    valid_step_freq=config.valid_step_freq,
    device=device
)

# Instanciar o Predictor
predictor = Predictor(
    model, 
    batch_size=config.batch_size, 
    n_z=config.test_n_z, 
    last_point_only=True,
    device=device
)

In [ ]:
start_time = time.time()

# Executa o loop de treinamento
train_metrics = trainer.fit(x_train, valid_portion=0.1)

print(f"\nTreinamento finalizado em {(time.time() - start_time):.2f} segundos.")
pprint(train_metrics)

In [ ]:
print("Iniciando inferência...")

# 1. Obter scores do conjunto de treino (Necessário para calibrar o POT)
# Isso pode demorar um pouco dependendo do tamanho do treino
train_score, train_z, _ = predictor.get_score(x_train)

# 2. Obter scores do conjunto de teste
test_score, test_z, pred_time = predictor.get_score(x_test)

# Se configurado para somar dimensões (padrão do OmniAnomaly é somar)
if not config.get_score_on_dim:
    # Garantir que é 1D array se já não foi somado no modelo
    if train_score.ndim > 1: train_score = np.sum(train_score, axis=-1)
    if test_score.ndim > 1: test_score = np.sum(test_score, axis=-1)

print(f"Tempo médio de predição: {pred_time:.4f}s")
print(f"Train Score shape: {train_score.shape}")
print(f"Test Score shape: {test_score.shape}")

In [ ]:
# 1. Preparação para o POT (Sem y_test real)
# Criamos um array de zeros apenas para satisfazer a assinatura da função pot_eval.
# Isso não afeta o valor do threshold, que depende apenas de train_score e test_score.
y_test_dummy = np.zeros_like(test_score)

print("Calculando limiar dinâmico (POT)...")
pot_result = pot_eval(
    train_score, 
    test_score, 
    y_test_dummy, 
    level=config.level
)
threshold = pot_result['pot-threshold']

# 2. Configuração do Gráfico
plt.figure(figsize=(20, 8))

# --- A. Score de Anomalia (Linha Azul) ---
plt.plot(test_score, label='Anomaly Score', color='#1f77b4', linewidth=1, alpha=0.8)

# --- B. Limiar de Decisão (Linha Tracejada Vermelha) ---
plt.axhline(y=threshold, color='orange', linestyle='--', linewidth=2, label=f'Threshold: {threshold:.4f}')

# --- C. Destaque das Anomalias Detectadas (Pontos Vermelhos) ---
# No OmniAnomaly, score < threshold indica anomalia (baixa probabilidade)
anomalies_idx = np.where(test_score < threshold)[0]

# Cosmética
plt.title(f"Monitoramento de Anomalias")
plt.xlabel("Tempo (Janelas)")
plt.ylabel("Score de Reconstrução (Log-Prob)")
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()

plt.show()

## MSCRED

Chuxu Zhang, Dongjin Song, Yuncong Chen, Xinyang Feng, Cristian Lumezanu, Wei Cheng, Jingchao Ni, Bo Zong, Haifeng Chen, and Nitesh V. Chawla. 2019. A deep neural network for unsupervised anomaly detection and diagnosis in multivariate time series data. In Proceedings of the Thirty-Third AAAI Conference on Artificial Intelligence and Thirty-First Innovative Applications of Artificial Intelligence Conference and Ninth AAAI Symposium on Educational Advances in Artificial Intelligence (AAAI'19/IAAI'19/EAAI'19), Vol. 33. AAAI Press, Article 174, 1409–1416. https://doi.org/10.1609/aaai.v33i01.33011409

In [ ]:
config = {
    'ts_type': 'node',
    'step_max': 5,
    'gap_time': 10,
    'win_size': [10, 30, 60],
}

def get_scaler(df_list):
    full_df = pd.concat(df_list, axis=0)
    
    # Transform to (Sensores, Tempo)
    data = np.array(full_df.values.T, dtype=np.float64)
    
    # Global min/max on training set
    max_val = np.max(data, axis=1, keepdims=True)
    min_val = np.min(data, axis=1, keepdims=True)
    
    return min_val, max_val

def generate_signature_matrix(df_block, config, scaler_params):
    step_max = config['step_max']
    gap_time = config['gap_time']
    win_size = config['win_size']
    
    # Preprocess
    data = np.array(df_block.values.T, dtype=np.float64)
    sensor_n = data.shape[0]
    total_time = data.shape[1]
    
    # MinMax Normalization
    min_val, max_val = scaler_params
    epsilon = 1e-6
    data = (data - min_val) / (max_val - min_val + epsilon)

    # Generate Signature Matrix
    data_all_scales = []
    for w in range(len(win_size)):
        matrix_list = []
        win = win_size[w]
        
        for t in range(0, total_time, gap_time):
            matrix_t = np.zeros((sensor_n, sensor_n))
            
            # t < win, generate zeros (padding), however the loop ignore this initial zeros
            if t >= win:
                segment = data[:, t - win : t]
                matrix_t = np.matmul(segment, segment.T) / win
            
            matrix_list.append(matrix_t)
        data_all_scales.append(matrix_list)

    # ConvLSTM
    X_block = []
    total_generated_steps = len(data_all_scales[0])
    num_scales = len(win_size)
    
    # Valid sequences when MAIOR JANELA + SEQUÊNCIA HISTÓRICA
    start_idx = (win_size[-1] // gap_time) + step_max

    for i in range(total_generated_steps):
        # Jump cold start
        if i < start_idx:
            continue
            
        sequence_matrices = [] 
        for step in range(step_max, 0, -1):
            idx = i - step
            multi_scale_step = []
            for scale_idx in range(num_scales):
                multi_scale_step.append(data_all_scales[scale_idx][idx])
            sequence_matrices.append(multi_scale_step)
        
        X_block.append(np.array(sequence_matrices))

    if len(X_block) > 0:
        return np.array(X_block)
    else:
        return np.empty((0)) # Retorna vazio se o bloco for muito pequeno


df_train_list = [
    df_train1,
    df_train2
]

df_test_list = [
    df_test
]

# Get scaler values
scaler = get_scaler(df_train_list)

X_train_list = []
for i, block in enumerate(df_train_list):
    print(f"df_train{i+1} shape {len(block)}...")
    X_proc = generate_signature_matrix(block, config, scaler)
    
    if X_proc.size > 0:
        X_train_list.append(X_proc)
        print(f" -> {X_proc.shape[0]} sequences.")

if X_train_list:
    X_train_final = np.concatenate(X_train_list, axis=0)
    print(f"\nFinal shape training data: {X_train_final.shape}")

X_test_list = []
for i, block in enumerate(df_test_list):
    X_proc = generate_signature_matrix(block, config, scaler)
    if X_proc.size > 0:
        X_test_list.append(X_proc)

if X_test_list:
    X_test_final = np.concatenate(X_test_list, axis=0)
    print(f"Final shape test data: {X_test_final.shape}")

In [ ]:
model_config = {
    'batch_size': 64,          # 32 - 128
    'sensor_n': 30,            # Att with data shape
    'win_size': [10, 30, 60],
    'step_max': 5,
    'learning_rate': 0.0002,
    'epochs': 5,
    'model_save_path': 'mscred_model/',
    'device': torch.device("cuda" if torch.cuda.is_available() else "cpu")
}

print(f"Using device: {model_config['device']}")

class MemoryDataset(Dataset):
    def __init__(self, data_array):
        """
        Args:
            data_array: Numpy array with shape (Samples, Steps, Scales, H, W)
        """
        self.data = data_array

    def __len__(self):
        return self.data.shape[0]

    def __getitem__(self, idx):
        # (Steps, Scales, H, W)
        sample = self.data[idx]
        
        # Convert to Tensor float32
        return torch.from_numpy(sample).float()

class ConvLSTMCell(nn.Module):
    def __init__(self, input_dim, hidden_dim, kernel_size, bias):
        super(ConvLSTMCell, self).__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.kernel_size = kernel_size
        self.padding = kernel_size[0] // 2, kernel_size[1] // 2
        self.bias = bias
        self.conv = nn.Conv2d(in_channels=self.input_dim + self.hidden_dim,
                              out_channels=4 * self.hidden_dim,
                              kernel_size=self.kernel_size,
                              padding=self.padding,
                              bias=self.bias)

    def forward(self, input_tensor, cur_state):
        h_cur, c_cur = cur_state
        combined = torch.cat([input_tensor, h_cur], dim=1)
        combined_conv = self.conv(combined)
        cc_i, cc_f, cc_o, cc_g = torch.split(combined_conv, self.hidden_dim, dim=1)
        i = torch.sigmoid(cc_i)
        f = torch.sigmoid(cc_f)
        o = torch.sigmoid(cc_o)
        g = torch.tanh(cc_g)
        c_next = f * c_cur + i * g
        h_next = o * torch.tanh(c_next)
        return h_next, c_next

    def init_hidden(self, batch_size, image_size, device):
        height, width = image_size
        return (torch.zeros(batch_size, self.hidden_dim, height, width, device=device),
                torch.zeros(batch_size, self.hidden_dim, height, width, device=device))

class TemporalAttention(nn.Module):
    def __init__(self):
        super(TemporalAttention, self).__init__()

    def forward(self, hidden_states):
        # hidden_states: [batch, steps, channels, H, W]
        last_step = hidden_states[:, -1, ...]
        steps = hidden_states.shape[1]
        weights = []
        for k in range(steps):
            curr_step = hidden_states[:, k, ...]
            # Eq (4) Dot Product
            dot_product = torch.sum(curr_step * last_step, dim=(1, 2, 3)) 
            weights.append(dot_product / steps)
        
        weights = torch.stack(weights, dim=1)
        weights = F.softmax(weights, dim=1).unsqueeze(2).unsqueeze(3).unsqueeze(4)
        context = torch.sum(hidden_states * weights, dim=1)
        return context

class MSCRED(nn.Module):
    def __init__(self, sensor_n, scale_n, step_max):
        super(MSCRED, self).__init__()
        self.sensor_n = sensor_n
        
        # Encoder
        self.conv1 = nn.Conv2d(scale_n, 32, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=2, stride=2, padding=0)
        self.conv4 = nn.Conv2d(128, 256, kernel_size=2, stride=2, padding=0)
        
        # ConvLSTMs
        self.lstm1 = ConvLSTMCell(32, 32, (3,3), True)
        self.lstm2 = ConvLSTMCell(64, 64, (3,3), True)
        self.lstm3 = ConvLSTMCell(128, 128, (3,3), True)
        self.lstm4 = ConvLSTMCell(256, 256, (3,3), True)
        
        self.attention = TemporalAttention()
        
        # Decoder
        self.deconv4 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2, padding=0)
        self.deconv3 = nn.ConvTranspose2d(256, 64, kernel_size=2, stride=2, padding=0)
        self.deconv2 = nn.ConvTranspose2d(128, 32, kernel_size=3, stride=2, padding=1)
        self.deconv1 = nn.ConvTranspose2d(64, scale_n, kernel_size=3, stride=1, padding=1)
        
    def forward(self, x):
        b, steps, c, h, w = x.size()
        h1_list, h2_list, h3_list, h4_list = [], [], [], []
        
        # init hidden states in the first loop
        # Calculate expected shapes after convolutions for init_hidden
        dummy_in = torch.zeros(1, c, h, w, device=x.device)
        d1 = self.conv1(dummy_in)
        d2 = self.conv2(d1)
        d3 = self.conv3(d2)
        d4 = self.conv4(d3)
        
        state1 = self.lstm1.init_hidden(b, (d1.size(2), d1.size(3)), x.device)
        state2 = self.lstm2.init_hidden(b, (d2.size(2), d2.size(3)), x.device)
        state3 = self.lstm3.init_hidden(b, (d3.size(2), d3.size(3)), x.device)
        state4 = self.lstm4.init_hidden(b, (d4.size(2), d4.size(3)), x.device)

        for t in range(steps):
            input_t = x[:, t, :, :, :]
            enc1 = F.selu(self.conv1(input_t))
            enc2 = F.selu(self.conv2(enc1))
            enc3 = F.selu(self.conv3(enc2))
            enc4 = F.selu(self.conv4(enc3))
            
            h1, c1 = self.lstm1(enc1, state1)
            state1 = (h1, c1)
            h1_list.append(h1)
            
            h2, c2 = self.lstm2(enc2, state2)
            state2 = (h2, c2)
            h2_list.append(h2)
            
            h3, c3 = self.lstm3(enc3, state3)
            state3 = (h3, c3)
            h3_list.append(h3)
            
            h4, c4 = self.lstm4(enc4, state4)
            state4 = (h4, c4)
            h4_list.append(h4)
            
        h1_stack = torch.stack(h1_list, dim=1)
        h2_stack = torch.stack(h2_list, dim=1)
        h3_stack = torch.stack(h3_list, dim=1)
        h4_stack = torch.stack(h4_list, dim=1)
        
        attn1 = self.attention(h1_stack)
        attn2 = self.attention(h2_stack)
        attn3 = self.attention(h3_stack)
        attn4 = self.attention(h4_stack)
        
        dec4 = F.selu(self.deconv4(attn4))
        if dec4.shape[2:] != attn3.shape[2:]: dec4 = F.interpolate(dec4, size=attn3.shape[2:])
        dec4_concat = torch.cat([dec4, attn3], dim=1)
        
        dec3 = F.selu(self.deconv3(dec4_concat))
        if dec3.shape[2:] != attn2.shape[2:]: dec3 = F.interpolate(dec3, size=attn2.shape[2:])
        dec3_concat = torch.cat([dec3, attn2], dim=1)
        
        dec2 = F.selu(self.deconv2(dec3_concat))
        if dec2.shape[2:] != attn1.shape[2:]: dec2 = F.interpolate(dec2, size=attn1.shape[2:])
        dec2_concat = torch.cat([dec2, attn1], dim=1)
        
        output = F.selu(self.deconv1(dec2_concat))
        return output


def train_model(X_train, config):
    device = config['device']
    
    if X_train is None or len(X_train) == 0:
        print("X_train is empty!")
        return None

    # Dataset e DataLoader
    dataset = MemoryDataset(X_train)
    dataloader = DataLoader(dataset, batch_size=config['batch_size'], shuffle=True)
    
    # Automatically identify dimensions
    # Expected Shape: [Samples, Steps, Scales, H, W]
    sample_shape = X_train.shape
    scale_n = sample_shape[2]
    sensor_n = sample_shape[3] # H = W = sensors
    
    print(f"Init training process... \n(Scales: {scale_n}, Sensors: {sensor_n})")

    model = MSCRED(sensor_n=sensor_n, scale_n=scale_n, step_max=config['step_max']).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=config['learning_rate'])
    criterion = nn.MSELoss()
    
    os.makedirs(config['model_save_path'], exist_ok=True)
    
    model.train()
    for epoch in range(config['epochs']):
        epoch_loss = 0
        start_time = time.time()
        
        for i, batch_x in enumerate(dataloader):
            batch_x = batch_x.to(device)
            target = batch_x[:, -1, :, :, :] # Target é reconstruir a última matriz
            
            optimizer.zero_grad()
            reconstruction = model(batch_x)
            
            loss = criterion(reconstruction, target)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            
        avg_loss = epoch_loss / len(dataloader)
        print(f"Epoch [{epoch+1}/{config['epochs']}] Loss: {avg_loss:.6f} Time: {time.time()-start_time:.2f}s")
        
    # Save final model
    # torch.save(model.state_dict(), os.path.join(config['model_save_path'], "mscred_final.pth"))
    print("\n")
    return model

def test_model(X_test, model, config):
    device = config['device']
    
    if X_test is None or len(X_test) == 0:
        print("X_test vazio.")
        return None
        
    dataset = MemoryDataset(X_test)
    dataloader = DataLoader(dataset, batch_size=config['batch_size'], shuffle=False)
    
    model.eval()
    reconstructions = []
    
    with torch.no_grad():
        for batch_x in dataloader:
            batch_x = batch_x.to(device)
            recon = model(batch_x)
            reconstructions.append(recon.cpu().numpy())
            
    # Concat batches
    full_recon = np.concatenate(reconstructions, axis=0)
    return full_recon


if 'X_train_final' in locals() and 'X_test_final' in locals():
    # Train
    trained_model = train_model(X_train_final, model_config)
    
    # Test
    reconstructions = test_model(X_test_final, trained_model, model_config)
    
    # Residual Error
    # MSE between original input (last step) and the reconstruction
    # X_test shape: (N, Steps, Scales, H, W) -> Get the last step: X_test[:, -1]
    residuals = np.square(X_test_final[:, -1] - reconstructions)
    
else:
    print("X_train_final or X_test_final not found in local!")

### Threshold Mean/MSE

Score de anomalia é calculado pela média (np.mean) de todos os erros da matriz de correlação. Como o MSCRED trabalha com matrizes (e não com uma única série temporal), uma falha localizada — a quebra de correlação entre apenas um par de sensores — tende a ser diluída em meio a centenas de pares. Assim, mesmo com um erro elevado em um ponto específico, o valor médio pode aumentar muito pouco e permanecer abaixo do limiar, reduzindo a sensibilidade à detecção desse tipo de anomalia localizada.

In [ ]:
def calculate_mse_scores(X_origin, X_recon):
    # Ground Truth (Last sequence step)
    # Shape: (N, Scales, H, W)
    ground_truth = X_origin[:, -1, :, :, :]
    
    # Residuals Matrix
    residual_matrix = np.square(ground_truth - X_recon)
    
    # Score: Error mean of all sensors and scales
    scores = np.mean(residual_matrix, axis=(1, 2, 3))
    
    return scores

def detect_and_plot_anomalies(X_test, X_test_recon, X_train, X_train_recon, percentile=95):
    """
    Define threshold based on TRAIN ser and apply on TEST set.
    
    Args:
        X_test: Original data test
        X_test_recon: Reconstructed data test
        X_train: Original data train
        X_train_recon: Reconstructed data train
        percentile: Percentile of the TRAIN set to define the threshol (ex: 95, 99)
    """
    
    # Calculate train scores
    train_scores = calculate_mse_scores(X_train, X_train_recon)
    
    # Define threshold based on train set
    # The threshold is the value that covers X% of normal training data
    threshold = np.percentile(train_scores, percentile)
    print(f"Threshold (Percentile {percentile} on train set): {threshold:.6f}")
    print(f"Max Score train: {np.max(train_scores):.6f}")

    # Calculate test scores
    test_scores = calculate_mse_scores(X_test, X_test_recon)
    
    # Get anomaly indexes
    anomalies_indices = np.where(test_scores > threshold)[0]

    # Plotly visualization
    fig = go.Figure()

    # Score Line
    fig.add_trace(go.Scatter(
        y=test_scores,
        mode='lines',
        name='Score de Anomalia (Teste)',
        line=dict(color='blue', width=1),
        opacity=0.7
    ))

    # Threshold Line
    fig.add_hline(
        y=threshold, 
        line_dash="dash", 
        line_color="red"
    )

    # Plot Layout
    fig.update_layout(
        title='Detecção de Anomalias (Threshold calibrado via Treino)',
        xaxis_title='Timestamp (Amostras de Teste)',
        yaxis_title='Erro de Reconstrução (MSE)',
        template='plotly_white',
        hovermode="x unified",
        legend=dict(
            yanchor="top",
            y=0.99,
            xanchor="left",
            x=0.01
        )
    )
    fig.show()
    
    return test_scores, threshold


reconstructions_train = test_model(X_train_final, trained_model, model_config)
scores, threshold = detect_and_plot_anomalies(
    X_test=X_test_final, 
    X_test_recon=reconstructions, # reconstrução do teste
    X_train=X_train_final, 
    X_train_recon=reconstructions_train, # reconstrução do treino
    percentile=99 # Geralmente usa-se 99 ou 99.9 para evitar falsos positivos
)

### Threshold with paper method

Score de anomalia é obtido somando quantos elementos da matriz de erro ultrapassam um limiar mínimo (thred_broken). Em vez de diluir o erro médio, o método ignora pequenas variações normais (ruído) e foca diretamente na quantidade de correlações quebradas. Dessa forma, falhas localizadas — mesmo que ocorram em poucos pares de sensores — são destacadas de forma clara, tornando a detecção mais sensível e robusta.

In [ ]:
def compute_anomaly_scores(X_origin, X_recon, thred_broken):
    # Ground Truth (Last Step of sequence)
    ground_truth = X_origin[:, -1, :, :, :]
    
    # Residuals
    residual_matrix = np.square(ground_truth - X_recon)
    
    # Contagem de Elementos Quebrados (Soma sobre Scales, H, W)
    broken_elements_mask = residual_matrix > thred_broken
    anomaly_scores = np.sum(broken_elements_mask, axis=(1, 2, 3))
    
    return anomaly_scores

def evaluate_mscred_with_train_calibration(
        X_train, X_train_recon, 
        X_test, X_test_recon, 
        thred_broken=0.05, 
        alpha=1.5):
    """
    Avalia o modelo usando o dataset de TREINO para definir o limiar (calibração)
    e o dataset de TESTE para detecção, com visualização em Plotly.
    """
    
    # Train Scores (Calibração)
    train_scores = compute_anomaly_scores(X_train, X_train_recon, thred_broken)
    
    # Test Scores (Detecção)
    test_scores = compute_anomaly_scores(X_test, X_test_recon, thred_broken)
    
    # Threshold
    # O limiar é o Pior Score encontrado no Treino * Margem de Segurança
    max_valid_score = np.max(train_scores)
    threshold = max_valid_score * alpha
    
    print(f"Max Score Train: {max_valid_score}")
    print(f"Threshold (Max Score Train * {alpha}): {threshold:.4f}")
    
    # --- Visualização com Plotly ---
    fig = go.Figure()

    # Linha do Score de Teste
    fig.add_trace(go.Scatter(
        y=test_scores,
        mode='lines',
        name='Score (Teste)',
        line=dict(color='#1f77b4', width=1.5),
        opacity=0.8
    ))

    # Identificar índices de anomalia para destacar
    anom_indices = np.where(test_scores > threshold)[0]

    # Linha do Limiar
    fig.add_hline(
        y=threshold,
        line_dash="dash",
        line_color="red"
    )

    # Configurações do Layout
    fig.update_layout(
        title="Detecção de Anomalias MSCRED (Calibração via Treino)",
        xaxis_title="Tempo (Amostras de Teste)",
        yaxis_title="Score de Anomalia (Elementos Quebrados)",
        template='plotly_white',
        hovermode="x unified",
        legend=dict(
            yanchor="top",
            y=0.99,
            xanchor="left",
            x=0.01
        )
    )

    fig.show()
    
    return test_scores, threshold

print("Reconstruct data train to define threshold...")
reconstructions_train = test_model(X_train_final, trained_model, model_config)

# print("Reconstruct data test...")
# reconstructions_test = test_model(X_test_final, trained_model, model_config)

scores, thresh = evaluate_mscred_with_train_calibration(
    X_train=X_train_final,
    X_train_recon=reconstructions_train,
    X_test=X_test_final,
    X_test_recon=reconstructions,
    thred_broken=0.05,
    alpha=0.9 
)

### Threshold implemented with IQR

Nesta abordagem, o limiar de anomalia é definido a partir da distribuição dos scores de erro no conjunto de treino, utilizando o Intervalo Interquartil (IQR). Em vez de depender do valor máximo — que pode ser distorcido por outliers — o método calcula o limiar como Q3+k×IQR, tornando mais robusto ao ruído no treino e evitando que um único ponto extremo eleve excessivamente o limiar, preservando a sensibilidade para detectar anomalias mais sutis nos dados.

In [ ]:
def define_thred_broken_from_train(X_train, X_train_recon, coverage_percentile=99):
    # Ground Truth (Último passo da sequência)
    ground_truth = X_train[:, -1, :, :, :]
    
    # Calcular Resíduos Quadráticos (Pixel a Pixel)
    residual_matrix = np.square(ground_truth - X_train_recon)
    
    # Achatar para vetor 1D
    residuals_flat = residual_matrix.flatten()
    
    # Calcular o Percentil para cobrir X% dos casos normais
    thred_broken = np.percentile(residuals_flat, coverage_percentile)
    
    return thred_broken

def compute_anomaly_scores(X_origin, X_recon, thred_broken):
    # Ground Truth (Last Step of sequence)
    ground_truth = X_origin[:, -1, :, :, :]
    
    # Residuals
    residual_matrix = np.square(ground_truth - X_recon)
    
    # Contagem de Elementos Quebrados (Soma sobre Scales, H, W)
    broken_elements_mask = residual_matrix > thred_broken
    anomaly_scores = np.sum(broken_elements_mask, axis=(1, 2, 3))
    
    return anomaly_scores

def evaluate_mscred_robust(X_train, X_train_recon, X_test, X_test_recon, thred_broken=0.05, k=3.0):
    """
    Usa contagem de elementos quebrados (igual Opção 2), mas define o limiar usando Estatística Robusta (IQR) para ignorar outliers no treino.
    """
    
    # 1. Calcular Scores (Mesma lógica da Opção 2)
    train_scores = compute_anomaly_scores(X_train, X_train_recon, thred_broken)
    test_scores = compute_anomaly_scores(X_test, X_test_recon, thred_broken)
    
    # 2. Definição Robusta do Limiar (IQR)
    # Em vez de pegar o Max, pegamos os quartis
    q75, q25 = np.percentile(train_scores, [75 ,25])
    iqr = q75 - q25
    
    # Threshold = Q3 + (k * IQR)
    # k=1.5 é padrão estatístico para outliers leves, k=3.0 para outliers extremos
    threshold = q75 + (k * iqr)
    
    # Fallback de segurança: O threshold nunca deve ser menor que o máximo 'comum'
    # Se a distribuição for muito concentrada, usamos o max como piso
    # threshold = max(threshold, np.max(train_scores))

    print(f"Q3 Score: {q75:.4f} | IQR: {iqr:.4f}")
    print(f"Limiar Robusto (Q3 + {k}*IQR): {threshold:.4f}")

    # --- Visualização (Plotly) ---
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        y=test_scores, mode='lines', name='Score (Teste)',
        line=dict(color='#1f77b4', width=1.5), opacity=0.8
    ))

    # Anomalias
    anom_indices = np.where(test_scores > threshold)[0]

    fig.add_hline(y=threshold, line_dash="dash", line_color="red")

    fig.update_layout(
        title="Detecção Robusta (IQR)",
        xaxis_title="Tempo", yaxis_title="Score (Broken Elements)",
        template='plotly_white'
    )
    fig.show()
    
    return test_scores, threshold

print("Reconstruct data train to define threshold...")
reconstructions_train = test_model(X_train_final, trained_model, model_config)

# print("Reconstruct data test...")
# reconstructions_test = test_model(X_test_final, trained_model, model_config)

optimal_thred = define_thred_broken_from_train(
    X_train_final, 
    reconstructions_train, 
    coverage_percentile=99
)

scores, thresh = evaluate_mscred_robust(
    X_train=X_train_final,
    X_train_recon=reconstructions_train,
    X_test=X_test_final,
    X_test_recon=reconstructions,
    thred_broken=optimal_thred,
    k=3
)

## DAD Framework (CNN with two-stage LSTM-AE)

NIZAM, Hussain et al. Real-time deep anomaly detection framework for multivariate time-series data in industrial IoT. IEEE Sensors Journal, v. 22, n. 23, p. 22836-22849, 2022.

In [ ]:
# Configurações do Artigo (Tabela I) 
WINDOW_SIZE = 120
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
EPOCHS = 20 # Pode reduzir para testar rápido
DROPOUT_RATE = 0.1
NUM_CELLS = 50 

# Classe para criar as janelas deslizantes (Sliding Window)
class TimeSeriesDataset(Dataset):
    def __init__(self, data, window_size):
        self.data = torch.tensor(data, dtype=torch.float32)
        self.window_size = window_size

    def __len__(self):
        return len(self.data) - self.window_size + 1

    def __getitem__(self, idx):
        # O modelo é um Autoencoder, então Input (x) e Target (y) são iguais
        # Tentamos reconstruir a própria janela de entrada
        window = self.data[idx : idx + self.window_size]
        return window, window

def prepare_data(df_train, df_test, cols_to_use):
    # 1. Normalização (Eq. 14 do artigo) [cite: 346]
    scaler = MinMaxScaler()
    
    # Ajustar scaler apenas no treino para evitar data leakage
    train_scaled = scaler.fit_transform(df_train[cols_to_use])
    test_scaled = scaler.transform(df_test[cols_to_use])
    
    # 2. Criar Datasets e DataLoaders
    train_dataset = TimeSeriesDataset(train_scaled, WINDOW_SIZE)
    test_dataset = TimeSeriesDataset(test_scaled, WINDOW_SIZE)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    return train_loader, test_loader, len(cols_to_use), scaler

cols = df_train.select_dtypes(include=[np.number]).columns.tolist()
train_loader, test_loader, n_features, scaler = prepare_data(df_train, df_test, cols)

In [ ]:
class HybridAnomalyDetector(nn.Module):
    def __init__(self, n_features, window_size, dropout_rate=0.1):
        super(HybridAnomalyDetector, self).__init__()
        self.window_size = window_size
        self.n_features = n_features
        
        # --- 1. CNN Layer (Extração de Features Espaciais) ---
        # Baseado na Tabela I 
        # Entrada esperada pela Conv1d: (Batch, Channels/Features, Sequence_Length)
        
        self.cnn_block = nn.Sequential(
            # Conv1D (None, 15, 64) - Kernel/Stride configurados para reduzir dimensão
            nn.Conv1d(in_channels=n_features, out_channels=64, kernel_size=2, stride=2, padding=0),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=1), # (None, 14, 64)
            nn.Dropout(dropout_rate),
            
            # Conv1D (None, 14, 32)
            nn.Conv1d(in_channels=64, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=1), # (None, 13, 32)
            nn.Dropout(dropout_rate)
        )
        
        # Cálculo dinâmico do tamanho da saída da CNN para entrar na LSTM
        # Com Window=30, a saída após as convs acima terá length approx 13
        self.feature_map_len = 13 
        self.cnn_output_channels = 32
        
        # --- 2. Two-Stage LSTM Autoencoder (Paralelo) ---
        # O artigo cita dois LSTMs em paralelo para variações curtas e longas 
        
        # Estágio 1: Variações Longas
        self.lstm_stage1 = nn.LSTM(
            input_size=self.cnn_output_channels, 
            hidden_size=NUM_CELLS, 
            batch_first=True,
            bidirectional=False
        )
        
        # Estágio 2: Variações Curtas
        self.lstm_stage2 = nn.LSTM(
            input_size=self.cnn_output_channels, 
            hidden_size=NUM_CELLS, 
            batch_first=True,
            bidirectional=False
        )
        
        # --- 3. Dense / Output Layer ---
        # Reconstrói para o formato original (Window, Features)
        # Combinamos as saídas dos dois estágios (Concatenando ou Somando)
        # Aqui concatenamos para preservar informação de ambos os "experts"
        self.decoder_dense = nn.Sequential(
            nn.Linear(NUM_CELLS * 2, 64), # Tabela I cita Dense 64 
            nn.ReLU(),
            # Camada final para projetar de volta ao tamanho original da janela
            nn.Linear(64, window_size * n_features) 
        )

    def forward(self, x):
        batch_size = x.size(0)
        
        # x shape: (Batch, Window, Features)
        
        # 1. Passar pela CNN
        # PyTorch Conv1d quer (Batch, Features, Window)
        x_cnn_in = x.permute(0, 2, 1) 
        features = self.cnn_block(x_cnn_in) 
        
        # features shape: (Batch, 32, 13)
        
        # 2. Preparar para LSTM
        # LSTM quer (Batch, Seq_Len, Features/Channels)
        lstm_in = features.permute(0, 2, 1)
        
        # 3. Two-Stage LSTM (Paralelo)
        # Pegamos o último hidden state (_h) para representar a sequência comprimida
        _, (hidden_stage1, _) = self.lstm_stage1(lstm_in)
        _, (hidden_stage2, _) = self.lstm_stage2(lstm_in)
        
        # hidden shape: (1, Batch, Hidden_Size) -> removemos dim 1
        h1 = hidden_stage1[-1]
        h2 = hidden_stage2[-1]
        
        # 4. Combinar Estágios
        combined = torch.cat((h1, h2), dim=1) # (Batch, 100)
        
        # 5. Reconstrução
        reconstruction = self.decoder_dense(combined)
        
        # Reshape para (Batch, Window, Features) para calcular Loss
        reconstruction = reconstruction.view(batch_size, self.window_size, self.n_features)
        
        return reconstruction

In [ ]:
# Dispositivo (GPU se disponível)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

model = HybridAnomalyDetector(n_features=n_features, window_size=WINDOW_SIZE).to(device)

# Função de Perda e Otimizador
criterion = nn.MSELoss() # Baseado na Eq 16 (reconstruction error) [cite: 431]
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE) # 

def train_model(model, train_loader, val_loader, epochs):
    model.train()
    for epoch in range(epochs):
        train_loss = 0.0
        
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            
            optimizer.zero_grad()
            
            # Forward
            outputs = model(batch_x)
            
            # Loss: Diferença entre a entrada original e a reconstruída
            loss = criterion(outputs, batch_y)
            
            # Backward
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            
        # Validação simples a cada 10 épocas
        if (epoch + 1) % 10 == 0:
            val_loss = 0.0
            model.eval()
            with torch.no_grad():
                for val_x, val_y in val_loader:
                    val_x, val_y = val_x.to(device), val_y.to(device)
                    v_out = model(val_x)
                    val_loss += criterion(v_out, val_y).item()
            model.train()
            print(f"Epoch {epoch+1}/{epochs}, Train Loss: {train_loss/len(train_loader):.4f}, Val Loss: {val_loss/len(val_loader):.4f}")

# Exemplo de execução:
train_model(model, train_loader, test_loader, EPOCHS)

In [ ]:
def get_anomaly_scores(model, loader):
    model.eval()
    losses = []
    with torch.no_grad():
        for batch_x, _ in loader:
            batch_x = batch_x.to(device)
            reconstruction = model(batch_x)
            
            # Calcular erro MAE ou MSE por amostra (Eq 17/18) [cite: 434]
            # Aqui calculamos a média do erro absoluto por janela
            error = torch.mean(torch.abs(reconstruction - batch_x), dim=(1, 2))
            losses.extend(error.cpu().numpy())
            
    return np.array(losses)

train_scores = get_anomaly_scores(model, train_loader)
test_scores = get_anomaly_scores(model, test_loader)

# Definir Threshold (ex: média + 3*std do treino)
threshold = np.mean(train_scores) + 5 * np.std(train_scores)

In [ ]:
def plot_anomaly_results(test_scores, threshold, title="Detecção de Anomalias: Erro de Reconstrução vs Limiar"):
    """
    Plota os scores de anomalia (erro de reconstrução), a linha de limiar e destaca as anomalias.
    
    Args:
        test_scores (array-like): Vetor com os erros de reconstrução do conjunto de teste.
        threshold (float): Valor do limiar para corte.
    """
    # Criar DataFrame para facilitar manipulação
    df_plot = pd.DataFrame({'score': test_scores})
    df_plot['index'] = df_plot.index
    
    # Identificar anomalias (Score > Threshold)
    anomalies = df_plot[df_plot['score'] > threshold]

    # Criar a figura
    fig = go.Figure()

    # 1. Linha do Erro de Reconstrução (Score)
    fig.add_trace(go.Scatter(
        x=df_plot['index'], 
        y=df_plot['score'],
        mode='lines',
        name='Erro de Reconstrução',
        line=dict(color='blue', width=1.5),
        opacity=0.8
    ))

    # 2. Linha do Limiar (Threshold)
    fig.add_trace(go.Scatter(
        x=[df_plot['index'].min(), df_plot['index'].max()],
        y=[threshold, threshold],
        mode='lines',
        name='Limiar (Threshold)',
        line=dict(color='red', width=2, dash='dash')
    ))

    # Layout
    fig.update_layout(
        title=title,
        xaxis_title='Índice da Janela Temporal',
        yaxis_title='Erro de Reconstrução (MSE/MAE)',
        template='plotly_white',
        hovermode='x unified',
        legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
    )
    fig.show()

plot_anomaly_results(test_scores, threshold)

## Attention-Based Deep Convolutional Autoencoding Prediction Network (AT-DCAEP)

Liu, W., Yan, L., Ma, N., Wang, G., Ma, X., Liu, P., & Tang, R. (2024). Unsupervised deep anomaly detection for industrial multivariate time series data. Applied Sciences, 14(2), 774.

In [ ]:
class ExternalAttention(nn.Module):
    """
    Implementação do External Attention baseada na referência [36] citada no artigo.
    "External attention only uses two linear layers... memory unit M"[cite: 286, 289].
    """
    def __init__(self, d_model, S=64):
        super(ExternalAttention, self).__init__()
        self.mk = nn.Linear(d_model, S, bias=False) # Primeira camada linear (Memory Key)
        self.mv = nn.Linear(S, d_model, bias=False) # Segunda camada linear (Memory Value)
        self.softmax = nn.Softmax(dim=1) # Normalization [cite: 290]

    def forward(self, x):
        # Eq 16: A = Norm(O * M_k^T)
        attn = self.mk(x) # (Batch, Time, S)
        attn = self.softmax(attn) # Normalização da atenção
        
        # Eq 17: P_out = A * M_v
        attn = attn / (1e-9 + attn.sum(dim=-1, keepdim=True)) # Normalização adicional
        out = self.mv(attn) # (Batch, Time, d_model)
        return out

class CharacterizationNetwork(nn.Module):
    """
    Rede de Caracterização baseada em Autoencoder Convolucional + Multi-Head Attention.
    """
    def __init__(self, n_features, window_size=11):
        super(CharacterizationNetwork, self).__init__()
        
        # Encoder
        self.enc_conv1 = nn.Conv1d(n_features, 32, kernel_size=3, stride=2, padding=1) # [cite: 206-209]
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2) # [cite: 207]
        self.enc_conv2 = nn.Conv1d(32, 16, kernel_size=3, stride=2, padding=1) # [cite: 161]
        
        # Low-Dimensional Feature Extraction [cite: 212]
        self.multihead_attn = nn.MultiheadAttention(embed_dim=16, num_heads=4, batch_first=True)
        
        # Decoder
        self.dec_conv1 = nn.ConvTranspose1d(16, 32, kernel_size=3, stride=2, padding=1, output_padding=0) # [cite: 158]
        self.dec_conv2 = nn.ConvTranspose1d(32, n_features, kernel_size=3, stride=2, padding=1, output_padding=1) # [cite: 155]
        
        self.window_size = window_size

    def forward(self, x):
        x = x.permute(0, 2, 1) # (Batch, Features, Window)
        
        # --- ENCODER ---
        x_enc = self.relu(self.enc_conv1(x))
        x_enc = self.dropout(x_enc)
        D = self.relu(self.enc_conv2(x_enc))
        
        # --- LOW-DIMENSIONAL FEATURE EXTRACTION ---
        D_perm = D.permute(0, 2, 1) 
        D_prime, _ = self.multihead_attn(D_perm, D_perm, D_perm)
        D_prime = D_prime.permute(0, 2, 1)
        
        # --- DECODER ---
        x_dec = self.relu(self.dec_conv1(D_prime))
        x_dec = self.dropout(x_dec)
        S_prime = self.dec_conv2(x_dec)
        
        # Ajuste fino de tamanho (interpolação)
        if S_prime.shape[2] != self.window_size:
            S_prime = F.interpolate(S_prime, size=self.window_size, mode='linear', align_corners=False)
            
        S_prime = S_prime.permute(0, 2, 1) # (Batch, Window, Feat)
        return S_prime, D_prime

class PredictionNetwork(nn.Module):
    """
    Rede de Predição baseada em Atenção (LSTM + External Attention).
    """
    def __init__(self, n_features, latent_dim=16):
        super(PredictionNetwork, self).__init__()
        
        # Downsampling [cite: 240]
        self.conv_down = nn.Conv1d(n_features, latent_dim, kernel_size=1, stride=4) 
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)
        
        # Upsampling [cite: 250]
        self.conv_up = nn.ConvTranspose1d(latent_dim, latent_dim, kernel_size=1, stride=4, output_padding=0)
        
        # LSTM [cite: 254]
        self.lstm = nn.LSTM(input_size=latent_dim, hidden_size=32, batch_first=True)
        
        # External Attention [cite: 285]
        self.external_attn = ExternalAttention(d_model=32, S=64)
        
        self.final_proj = nn.Linear(32, n_features)

    def forward(self, R, D_prime, target_len):
        R_in = R.permute(0, 2, 1)
        
        # Eq 8: D' (fused) = Downsample(R) + D'
        R_down = self.relu(self.conv_down(R_in))
        
        if R_down.shape[2] != D_prime.shape[2]:
             R_down = F.interpolate(R_down, size=D_prime.shape[2])
             
        R_prime = R_down + D_prime # Fusão [cite: 246, 251]
        
        # Eq 9: E = Upsample(R')
        E = self.conv_up(R_prime)
        
        if E.shape[2] != target_len:
            E = F.interpolate(E, size=target_len, mode='linear', align_corners=False)
            
        E = E.permute(0, 2, 1)
        
        # LSTM & External Attention
        lstm_out, _ = self.lstm(E)
        lstm_out = self.dropout(lstm_out)
        attn_out = self.external_attn(lstm_out) # Eq 17
        
        P_out = self.final_proj(attn_out)
        return P_out

class AT_DCAEP(nn.Module):
    def __init__(self, n_features, window_size=11):
        super(AT_DCAEP, self).__init__()
        self.characterization_net = CharacterizationNetwork(n_features, window_size)
        self.prediction_net = PredictionNetwork(n_features)
        self.window_size = window_size

    def forward(self, S):
        S_prime, D_prime = self.characterization_net(S)
        
        # Calcular Erro de Reconstrução R [cite: 147]
        R = (S - S_prime) ** 2 
        
        P_out = self.prediction_net(R, D_prime, self.window_size)
        return S_prime, P_out, R

# ==========================================
# 3. FUNÇÕES AUXILIARES
# ==========================================

def loss_function(S, S_prime, R, P_out):
    """ Eq 19: L = ||S - S'||2 + ||R - P_out||2 [cite: 302] """
    loss_reconstruction = F.mse_loss(S_prime, S)
    loss_prediction = F.mse_loss(P_out, R)
    return loss_reconstruction + loss_prediction

def create_sliding_window(data, window_size):
    samples = []
    for i in range(len(data) - window_size + 1):
        samples.append(data[i : i + window_size])
    return np.array(samples)

def prepare_data(df_train, df_test, window_size=11):
    train_data = df_train.values
    test_data = df_test.values

    # Normalização Min-Max baseada APENAS no Treino [cite: 125]
    epsilon = 1e-5
    min_val = np.min(train_data, axis=0)
    max_val = np.max(train_data, axis=0)

    train_norm = (train_data - min_val) / (max_val - min_val + epsilon)
    test_norm = (test_data - min_val) / (max_val - min_val + epsilon)

    X_train = create_sliding_window(train_norm, window_size)
    X_test = create_sliding_window(test_norm, window_size)

    tensor_train = torch.FloatTensor(X_train)
    tensor_test = torch.FloatTensor(X_test)
    return tensor_train, tensor_test

def train_model(model, train_loader, num_epochs=50, learning_rate=1e-3):
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    print(f"Iniciando treinamento em: {device}")

    model.train()

    # Loop de Treinamento (Algoritmo 1) [cite: 325]
    for epoch in range(num_epochs):
        start_time = time.time()
        total_loss = 0
        
        for batch_idx, (data,) in enumerate(train_loader):
            data = data.to(device)
            optimizer.zero_grad()
            
            # Forward & Loss
            S_prime, P_out, R = model(data)
            loss = loss_function(data, S_prime, R, P_out) # [cite: 302]
            
            # Backward
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        end_time = time.time()
        print(f"Epoch [{epoch+1}/{num_epochs}] | Loss: {avg_loss:.6f} | Tempo: {end_time - start_time:.2f}s")

    print("Treinamento concluído.")
    return model

In [ ]:
# Parâmetros
batch_size = 128
window_size = 60
n_features = df_train.shape[1]
EPOCHS = 10
LR = 1e-3

print(f"Configuração: Features={n_features}, Window={window_size}, Batch={batch_size}")

# Preparação dos Dados
tensor_train, tensor_test = prepare_data(df_train, df_test, window_size=window_size)
train_dataset = TensorDataset(tensor_train)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# Inicialização do Modelo
model = AT_DCAEP(n_features=n_features, window_size=window_size)

print("\n--- Iniciando Loop de Treinamento ---")
# Treinando por poucas épocas apenas para demonstração
trained_model = train_model(model, train_loader, num_epochs=EPOCHS, learning_rate=LR)

In [ ]:
def evaluate_and_plot(model, tensor_train, tensor_test, alpha=0.3, beta=0.7, percentile=99):
    """
    1. Calcula o Threshold usando dados de TREINO (assumindo que são majoritariamente normais).
    2. Avalia e plota os dados de TESTE usando esse threshold fixo.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.eval() # Modo de avaliação
    
    print("1. Processando dados de TREINO para definir o Threshold...")
    train_dataset = TensorDataset(tensor_train)
    # Batch size pode ser grande aqui pois não estamos treinando
    train_loader = DataLoader(train_dataset, batch_size=256, shuffle=False)
    
    train_scores = []
    
    with torch.no_grad():
        for (data,) in train_loader:
            data = data.to(device)
            
            # Forward Pass
            S_prime, P_out, R = model(data)
            
            # [cite_start]Eq 20 [cite: 318]
            loss_rec = torch.norm(data - S_prime, p=2, dim=(1,2))
            loss_pred = torch.norm(R - P_out, p=2, dim=(1,2))
            
            score = alpha * loss_rec + beta * loss_pred
            train_scores.extend(score.cpu().numpy())
            
    # Define o threshold baseado na distribuição de treino
    # Se o treino é "limpo", 99% a 99.9% é um bom ponto de corte para ruído
    threshold = np.percentile(train_scores, percentile)
    print(f"   -> Threshold definido (Percentil {percentile}% do Treino): {threshold:.4f}")

    print("2. Calculando scores de anomalia no conjunto de TESTE...")
    test_dataset = TensorDataset(tensor_test)
    test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)
    
    test_anomaly_scores = []
    
    with torch.no_grad():
        for (data,) in test_loader:
            data = data.to(device)
            
            S_prime, P_out, R = model(data)
            
            loss_rec = torch.norm(data - S_prime, p=2, dim=(1,2))
            loss_pred = torch.norm(R - P_out, p=2, dim=(1,2))
            
            score = alpha * loss_rec + beta * loss_pred
            test_anomaly_scores.extend(score.cpu().numpy())
            
    test_anomaly_scores = np.array(test_anomaly_scores)
    
    # Identifica anomalias no teste usando o threshold do treino
    anomalies = test_anomaly_scores > threshold
    print(f"   -> Total de janelas anômalas detectadas no teste: {np.sum(anomalies)}")


    fig = go.Figure()
    # Linha dos Scores de Teste
    fig.add_trace(go.Scatter(
        y=test_anomaly_scores,
        mode='lines',
        name='Score (Teste)',
        line=dict(color='blue', width=1),
        opacity=0.8
    ))

    # Pontos de Anomalia
    anomaly_indices = np.where(anomalies)[0]
    
    # Linha do Limiar
    fig.add_hline(
        y=threshold,
        line_dash="dash",
        line_color="red"
    )

    fig.update_layout(
        title="Detecção de Anomalias (Threshold baseado no Treino)",
        xaxis_title="Índice da Janela Temporal (Teste)",
        yaxis_title="Anomaly Score",
        template="plotly_white",
        hovermode="x unified",
        legend=dict(orientation="h", y=1.02, x=1, xanchor="right")
    )

    fig.show()
    
    return test_anomaly_scores, threshold

scores, thresh = evaluate_and_plot(model, tensor_train, tensor_test, alpha=0.3, beta=0.7, percentile=99)

## An Attention-Based ConvLSTM Autoencoder with Dynamic Thresholding for Unsupervised Anomaly Detection in Multivariate Time Series (ACLAE-DT)

Tayeh, T., Aburakhia, S., Myers, R., & Shami, A. (2022). An attention-based ConvLSTM autoencoder with dynamic thresholding for unsupervised anomaly detection in multivariate time series. Machine Learning and Knowledge Extraction, 4(2), 350-370.

## TranAD: Deep Transformer Networks for Anomaly Detection in Multivariate Time Series Data

Tuli, S., Casale, G., & Jennings, N. R. (2022). Tranad: Deep transformer networks for anomaly detection in multivariate time series data. arXiv preprint arXiv:2201.07284.

## Robust Unsupervised Anomaly Detection With Variational Autoencoder in Multivariate Time Series Data (MSCVAE)

Yokkampon, U., Mowshowitz, A., Chumkamon, S., & Hayashi, E. (2022). Robust unsupervised anomaly detection with variational autoencoder in multivariate time series data. IEEE Access, 10, 57835-57849.